# Data Preprocessing and Feature Engineering

Shared by both Phase 2 problems:
- **Classification, conditional on delivery**: will the order arrive after Olist's estimated delivery calendar date (`is_late`)? Delivery on that date counts as on time. Cancelled, unavailable and unfinished orders are not labelled late.
- **Regression, conditional on delivery**: how many days from purchase to delivery (`delivery_days`)?

This notebook builds one order-level table with the features, both targets, a primary time-based train/test split, a secondary random benchmark, and their validation folds, and saves it to `data/`. Future classification and regression notebooks load that file, so they use the same orders, features and partitions within each experiment.

**Preprocessing flow**

| Stage | What is done | Where |
|---|---|---|
| 1. Data cleaning | Missing values, invalid values and outliers (kept), duplicates | Here |
| 2. Data reduction | Attribute reduction (e.g. coordinates to distance), row reduction (items to orders) | Here |
| 3. Data transformation | Feature construction, then normalisation, discretisation and encoding decisions | Construction here; normalisation and encoding fitted inside each model `Pipeline` |
| 4. Feature engineering strategy | Final feature set, rationale from Phase 1, initial leakage screen | Here |
| 5. Split and CV folds | Primary chronological 80/20 holdout with 5 expanding validation windows; secondary random 80/20 split with 5 stratified folds | Here |

**Leakage rules**
1. The intended prediction moment is immediately after checkout and payment submission, before approval or dispatch. Approval, carrier, delivery, status and review fields are excluded as inputs; availability of all remaining fields still needs an as-of-checkout audit.
2. One row per order, so items of one order never sit on both sides of the split.
3. No imputation statistics, scaling parameters, target encoding or seller late rates are fitted here. Fitted model preparation belongs inside each training fold. ZIP-prefix centroids are currently computed from the full geolocation table; verify their as-of availability in the leakage audit.

**Design principles**
1. **Checkout-time only.** The model runs immediately after checkout and payment submission, before approval or dispatch. Every input must be known at that moment; Section 4 currently screens column names, and a source-timing audit remains open.
2. **Evidence-led.** Each feature group is motivated by a Phase 1 finding or by the literature review, not added speculatively.
3. **Compact and interpretable.** 18 features that a stakeholder can understand, instead of hundreds of raw columns, to support the interpretability analysis.
4. **Family-specific preparation inside Pipelines.** Shared, data-independent work happens here; anything fitted (imputation, scaling, encoding) happens per model family on training folds.
5. **Selection in the model notebooks.** Feature importance and permutation importance decide whether weak features (e.g. weight) are pruned.
The evaluation rationale and limits are in Section 5 and the Summary below. [Phase 2 assumptions and preprocessing decisions](../../docs/phase2_assumptions.md) are logged for the report.


## For Model Notebooks: Read This First

**Output files** (saved in Section 5.3)
- `data/phase2_order_table.csv`: 96,470 rows, one per delivered order.
- `data/phase2_feature_spec.json`: feature lists, target names, split cutoff, `random_state` and `n_folds`.

**Columns**

| Column | Type | Meaning | Missing | Model input? |
|---|---|---|---|---|
| `order_id` | text | Order identifier | 0 | No |
| `order_purchase_timestamp` | datetime (text) | Purchase time, kept for reference | 0 | No |
| `promised_days` | numeric | Days from purchase to Olist's estimated delivery date (shown at checkout) | 0 | Yes |
| `distance_km` | numeric | Distance between customer and furthest seller, ZIP-prefix centroids | 477 | Yes |
| `same_state` | 0/1 | Every seller is in the customer's state | 0 | Yes |
| `purchase_hour` | 0-23 | Hour of purchase (cyclical) | 0 | Yes |
| `n_items`, `n_products`, `n_sellers` | count | Items, distinct products and distinct sellers in the order | 0 | Yes |
| `total_price`, `total_freight` | R$ | Sums over items | 0 | Yes |
| `total_weight_g`, `total_volume_cm3` | numeric | Sums over items | 16 each | Yes |
| `max_installments` | count | Highest instalment count across payments | 1 | Yes |
| `customer_state`, `seller_state` | category (27, 22) | Customer's state; furthest seller's state | 0 | Yes |
| `product_category` | category (72) | Category of the largest item by volume; `unknown` if missing | 0 (1,376 `unknown`) | Yes |
| `payment_type` | category (5) | Primary payment (lowest `payment_sequential`); `unknown` if none | 0 (1 `unknown`) | Yes |
| `purchase_dayofweek`, `purchase_month` | category (7, 12) | Day name, month name | 0 | Yes |
| `is_late` | 0/1 | **Classification target**: delivered after the estimated calendar date (6.77% late); same-date delivery is on time | 0 | No |
| `delivery_days` | numeric | **Regression target**: purchase to delivery in fractional days, right-skewed | 0 | No |
| `outcome_available_at` | datetime (text) | Actual delivery time; split metadata only | 0 | No |
| `split` | `train` / `test` / `unavailable_at_cutoff` | Primary time-based split | 0 | No |
| `cv_fold` | -1 / 0-4 | Primary training: initial history (-1), then five ordered validation blocks; empty elsewhere | varies | No |
| `random_split` | `train` / `test` | Secondary seeded stratified random 80/20 split | 0 | No |
| `random_cv_fold` | 0-4 | Secondary stratified fold for random training rows; empty for random test | 19,294 (test) | No |



**Already done in this table**
- Population: delivered orders with both a delivery and an estimated delivery timestamp.
- Invalid geolocation points removed; duplicate points merged into median ZIP-prefix centroids.
- Items and payments aggregated to one row per order.
- Categorical gaps filled with `unknown`; binary features encoded as 0/1.
- Candidate purchase-time features; the post-purchase column-name screen passed, while the full availability audit remains open. Redundant `purchase_is_weekend` was removed.
- Outliers kept on purpose (long deliveries, expensive or heavy orders are real).

**Not done: these should be finished inside the model `Pipeline`, fitted on training folds only**

| Step | Columns | Model families |
|---|---|---|
| Impute missing values (median) | `distance_km`, `total_weight_g`, `total_volume_cm3`, `max_installments` | A required; B and C optional (sklearn trees, HistGradientBoosting and CatBoost accept `NaN`) |
| Scale (`StandardScaler`) | Numeric features | A |
| Encode categories | 6 categorical features | A one-hot; B ordinal; C CatBoost native (Section 3.4) |
| Sine/cosine of `purchase_hour` | `purchase_hour` | A (Section 3.4) |
| `log1p` target via `TransformedTargetRegressor` | `delivery_days` | A, regression (Section 3.2) |
| Optional: `log1p` of skewed features, bin distance | `total_price`, `total_freight`, `total_weight_g`, `total_volume_cm3`, `distance_km` | A, keep only if CV improves (Sections 3.2, 3.3) |
| Class imbalance | `is_late` | Class weights and threshold tuning; report PR-AUC, not accuracy |
| Feature selection | All | Based on feature / permutation importance |

**Rules**
- Use `split` and the expanding folds for primary model selection in both tasks. The `random_*` columns are a separate secondary experiment.
- Do not touch primary test rows until the final evaluation. Exclude `unavailable_at_cutoff` from primary fitting.
- Do not fit anything (imputer, scaler, encoder, target statistics) outside a `Pipeline`.

**Load the primary experiment**

```python
import json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "olist_orders_dataset.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or a folder beneath it; root data/ was not found")
DATA_DIR = PROJECT_ROOT / "data"
table = pd.read_csv(DATA_DIR / "phase2_order_table.csv", parse_dates=["order_purchase_timestamp", "outcome_available_at"])
spec = json.loads((DATA_DIR / "phase2_feature_spec.json").read_text())
FEATURES = spec["numeric_features"] + spec["categorical_features"]
TARGET = spec["classification_target"]  # or spec["regression_target"]
train = table.loc[table["split"].eq("train")].reset_index(drop=True)
test = table.loc[table["split"].eq("test")].reset_index(drop=True)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

# Pass this list as cv= to cross_validate or GridSearchCV.
# At each validation start, use only earlier purchases with known outcomes.
cv = []
for fold in range(spec["n_folds"]):
    valid = np.flatnonzero(train["cv_fold"].eq(fold).to_numpy())
    start = train.loc[valid, "order_purchase_timestamp"].min()
    fit = np.flatnonzero(((train["order_purchase_timestamp"] < start) &
                           (train["outcome_available_at"] < start)).to_numpy())
    cv.append((fit, valid))
```

For the secondary benchmark, use a **separate fit**:

```python
from sklearn.model_selection import PredefinedSplit

random_train = table.loc[table["random_split"].eq("train")].reset_index(drop=True)
random_test = table.loc[table["random_split"].eq("test")].reset_index(drop=True)
random_cv = PredefinedSplit(random_train["random_cv_fold"].astype(int))
# Fit a fresh Pipeline on random_train[FEATURES] and random_train[TARGET].
# Evaluate that fit on random_test[FEATURES] and random_test[TARGET].
```

Do not use either split column or `outcome_available_at` as a feature.


## 0. Setup and Raw Data

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "olist_orders_dataset.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository or a folder beneath it; root data/ was not found")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_TABLE = DATA_DIR / "phase2_order_table.csv"
OUTPUT_SPEC = DATA_DIR / "phase2_feature_spec.json"

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_FOLDS = 5
TIME_CUTOFF = pd.Timestamp("2018-05-26")  # calendar-day boundary near the 80th purchase percentile
BRAZIL_LAT = (-34.0, 5.5)
BRAZIL_LNG = (-74.0, -34.5)

pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
TIMESTAMPS = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
orders_raw = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv", parse_dates=TIMESTAMPS)
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
geo = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")

raw_tables = {
    "orders": orders_raw, "order_items": items, "products": products, "sellers": sellers,
    "customers": customers, "payments": payments, "geolocation": geo,
}
pd.DataFrame({name: {"rows": len(t), "columns": t.shape[1]} for name, t in raw_tables.items()}).T

,rows,columns
orders,99441,8
order_items,112650,7
products,32951,9
sellers,3095,4
customers,99441,5
payments,103886,5
geolocation,1000163,5


## 1. Data Cleaning

### 1.1 Missing Values

**Missing targets: remove the rows.** An order needs both a delivery and an estimated delivery timestamp to have a label. Undelivered orders have no delivery-time or late-delivery label under the current targets, so the modelling population is delivered orders with both timestamps. The resulting predictions and evaluations are conditional on eventual delivery; see the assumptions log.

In [3]:
is_delivered = orders_raw["order_status"].eq("delivered")
has_dates = (orders_raw["order_delivered_customer_date"].notna()
             & orders_raw["order_estimated_delivery_date"].notna())
orders = orders_raw.loc[is_delivered & has_dates].copy()

pd.Series({
    "All orders": len(orders_raw),
    "Status = delivered": is_delivered.sum(),
    "Delivered with both timestamps (population)": len(orders),
}, name="orders").to_frame()

,orders
All orders,99441
Status = delivered,96478
Delivered with both timestamps (population),96470


**Missing attributes: keep the rows.** Dropping rows would lose orders, so missing attributes are handled by type:
- **Categorical:** fill with an explicit `unknown` category (no statistics needed, so no leakage). This applies to product category, and to payment type.
- **Numeric:** leave as `NaN` here and impute with the median inside each model `Pipeline`, so the median comes from training folds only.

In [4]:
population_items = items[items["order_id"].isin(orders["order_id"])]
population_products = products[products["product_id"].isin(population_items["product_id"])]
geo_zips = set(geo["geolocation_zip_code_prefix"])
population_customers = customers[customers["customer_id"].isin(orders["customer_id"])]
population_sellers = sellers[sellers["seller_id"].isin(population_items["seller_id"])]
has_payment = orders["order_id"].isin(payments["order_id"])
has_first_payment = orders["order_id"].isin(payments.loc[payments["payment_sequential"].eq(1), "order_id"])

pd.DataFrame([
    ("products", "product_category_name", population_products["product_category_name"].isna().sum(),
     "fill 'unknown' (categorical)"),
    ("products", "product_weight_g", population_products["product_weight_g"].isna().sum(),
     "NaN, median-imputed in Pipeline"),
    ("products", "length / height / width", population_products["product_length_cm"].isna().sum(),
     "NaN, median-imputed in Pipeline"),
    ("customers", "ZIP prefix not in geolocation",
     (~population_customers["customer_zip_code_prefix"].isin(geo_zips)).sum(),
     "distance NaN, median-imputed in Pipeline"),
    ("sellers", "ZIP prefix not in geolocation",
     (~population_sellers["seller_zip_code_prefix"].isin(geo_zips)).sum(),
     "distance NaN, median-imputed in Pipeline"),
    ("payments", "payment_type exist, but payment_sequential not numbered from 1",
     (has_payment & ~has_first_payment).sum(),
     "not missing: use the lowest-numbered row (Section 2.2)"),
    ("payments", "order without any payment row", (~has_payment).sum(),
     "fill 'unknown' (categorical)"),
], columns=["table", "attribute", "missing (population)", "handling"])

,table,attribute,missing (population),handling
0,products,product_category_name,584,fill 'unknown' (categorical)
1,products,product_weight_g,2,"NaN, median-imputed in Pipeline"
2,products,length / height / width,2,"NaN, median-imputed in Pipeline"
3,customers,ZIP prefix not in geolocation,264,"distance NaN, median-imputed in Pipeline"
4,sellers,ZIP prefix not in geolocation,7,"distance NaN, median-imputed in Pipeline"
5,payments,"payment_type exist, but payment_sequential not...",78,not missing: use the lowest-numbered row (Sect...
6,payments,order without any payment row,1,fill 'unknown' (categorical)


In [5]:
products = products.merge(translation, on="product_category_name", how="left")
products["product_category"] = products["product_category_name_english"].fillna("unknown")
print(f"Products with category 'unknown': {products['product_category'].eq('unknown').sum():,}")

Products with category 'unknown': 623


### 1.2 Invalid Values and Outliers

Two different cases:
- **Invalid values are removed.** Some geolocation points lie outside Brazil, a known Olist data error that produced distances up to about 8,700 km.
- **Outliers are kept.** Very long deliveries, expensive or heavy orders are real orders. The long tail of slow deliveries is exactly what the operations team needs to anticipate. Instead of removing outliers, their effect is handled in modelling: trees are robust to extreme values, and the regression target can be log-transformed for linear models (Section 3.2).

In [6]:
inside_brazil = (geo["geolocation_lat"].between(*BRAZIL_LAT)
                 & geo["geolocation_lng"].between(*BRAZIL_LNG))
print(f"Invalid geolocation points (outside Brazil) removed: {(~inside_brazil).sum():,}")
geo = geo.loc[inside_brazil]

Invalid geolocation points (outside Brazil) removed: 42


In [7]:
def iqr_outliers(values):
    q1, q3 = values.quantile([0.25, 0.75])
    return (values > q3 + 1.5 * (q3 - q1)).sum()


raw_duration_days = ((orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"])
                     .dt.total_seconds() / 86_400)
checks = {
    "purchase to delivery (days)": raw_duration_days,
    "item price": population_items["price"],
    "item freight": population_items["freight_value"],
    "product weight (g)": population_products["product_weight_g"].dropna(),
}
pd.DataFrame({
    name: {"median": v.median(), "max": v.max(), "above Q3 + 1.5 IQR": iqr_outliers(v),
           "share (%)": iqr_outliers(v) / len(v) * 100}
    for name, v in checks.items()
}).T.assign(decision="kept")

,median,max,above Q3 + 1.5 IQR,share (%),decision
purchase to delivery (days),10.22,209.63,4896.00,5.08,kept
item price,74.90,6735.00,8161.00,7.41,kept
item freight,16.26,409.68,11296.00,10.25,kept
product weight (g),700.00,40425.00,4442.00,13.79,kept


### 1.3 Duplicates

**Identify:** exact duplicate rows in each table, and whether each table's key is unique.

In [8]:
keys = {
    "orders": ["order_id"], "order_items": ["order_id", "order_item_id"], "products": ["product_id"],
    "sellers": ["seller_id"], "customers": ["customer_id"],
    "payments": ["order_id", "payment_sequential"], "geolocation": None,
}
pd.DataFrame({
    name: {
        "rows": len(table),
        "exact duplicate rows": table.duplicated().sum(),
        "key": " + ".join(keys[name]) if keys[name] else "none (many points per ZIP)",
        "key unique": table.duplicated(keys[name]).sum() == 0 if keys[name] else "-",
    }
    for name, table in raw_tables.items()
}).T

,rows,exact duplicate rows,key,key unique
orders,99441,0,order_id,True
order_items,112650,0,order_id + order_item_id,True
products,32951,0,product_id,True
sellers,3095,0,seller_id,True
customers,99441,0,customer_id,True
payments,103886,0,order_id + payment_sequential,True
geolocation,1000163,261831,none (many points per ZIP),-


**Remove/merge:** only the geolocation table has duplicates, in two forms:
1. **Exact duplicate rows** (same ZIP prefix, coordinates, city and state) are **removed**, so a point repeated many times does not pull the centroid towards itself.
2. **Many distinct points for the same ZIP prefix** describe the same entity, so they are **merged** into one centroid per prefix. The centroid is the **median** latitude and longitude rather than the mean: a few ZIP prefixes contain stray points far from the rest (e.g. in another state), and the median is not pulled by them.

Other repeated keys are not duplicates:
- Several items or payments per order are genuine one-to-many records. They are aggregated to order level in Section 2.2.
- Repeat customers (one `customer_unique_id` with several orders) are separate orders and stay.

In [9]:
n_before = len(geo)
geo = geo.drop_duplicates()
print(f"Exact duplicate geolocation rows removed: {n_before - len(geo):,}")

points_per_zip = geo.groupby("geolocation_zip_code_prefix").size()
print(f"Distinct points per ZIP prefix: median {points_per_zip.median():.0f}, max {points_per_zip.max():,}")

centroids = geo.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
print(f"{len(geo):,} distinct points merged into {len(centroids):,} ZIP-prefix centroids (median)")

Exact duplicate geolocation rows removed: 261,822
Distinct points per ZIP prefix: median 23, max 779
738,299 distinct points merged into 19,010 ZIP-prefix centroids (median)


## 2. Data Reduction

### 2.1 Attribute Reduction

**Location: up to 10 attributes become 2.** Customer and seller ZIP prefix, latitude, longitude and city are replaced by `distance_km` (Haversine distance between the seller and customer ZIP centroids) and `same_state`. Distance is created by feature construction (Section 3.1); its effect is attribute reduction. The raw coordinates carry no meaning for a model on their own, while distance captures the relationship Phase 1 found (delivery time triples across distance sextiles). Customer and seller state are kept as coarser location attributes.

In [10]:
def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    inner = (np.sin((lat2 - lat1) / 2) ** 2
             + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2)
    return 2 * 6371.0 * np.arcsin(np.sqrt(inner))


order_customers = orders[["order_id", "customer_id"]].merge(
    customers[["customer_id", "customer_zip_code_prefix", "customer_state"]], on="customer_id")
order_items = (
    items.merge(order_customers, on="order_id")  # keeps only population orders
    .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id", how="left")
)
customer_loc = centroids.reindex(order_items["customer_zip_code_prefix"]).to_numpy()
seller_loc = centroids.reindex(order_items["seller_zip_code_prefix"]).to_numpy()
order_items["distance_km"] = haversine_km(customer_loc[:, 0], customer_loc[:, 1],
                                          seller_loc[:, 0], seller_loc[:, 1])
order_items["same_state"] = order_items["seller_state"].eq(order_items["customer_state"])
order_items[["order_id", "seller_state", "customer_state", "distance_km", "same_state"]].head()

,order_id,seller_state,customer_state,distance_km,same_state
0,00010242fe8c5a6d1ba2dd792cb16214,SP,RJ,301.30,False
1,00018f77f2f0320c557190d7a144bdd3,SP,SP,588.54,True
2,000229ec398224ef6ca0657da4fc703e,MG,MG,312.96,True
3,00024acbcdf0a6daa1e931b038114c75,SP,SP,295.32,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,PR,SP,646.46,False


**Product size: 3 attributes become 1.** Length, height and width are combined into `volume_cm3`.

In [11]:
products["volume_cm3"] = (products["product_length_cm"] * products["product_height_cm"]
                          * products["product_width_cm"])
order_items = order_items.merge(
    products[["product_id", "product_category", "product_weight_g", "volume_cm3"]],
    on="product_id", how="left")
print(f"Item rows in population: {len(order_items):,}")

Item rows in population: 110,189


**Attributes dropped**

| Attribute(s) | Reason |
|---|---|
| `order_id`, `customer_id`, `customer_unique_id`, `seller_id`, `product_id` | Identifiers. Too many distinct values to generalise; a model would memorise IDs instead of learning patterns |
| ZIP prefixes, latitude, longitude, city (customer and seller) | Replaced by `distance_km` and `same_state`; city has thousands of values, so state is kept instead |
| `product_length_cm`, `product_height_cm`, `product_width_cm` | Replaced by `volume_cm3` |
| `product_name_lenght`, `product_description_lenght`, `product_photos_qty` | Describe the listing text, with no plausible link to delivery logistics |
| `order_approved_at`, `order_delivered_carrier_date`, `shipping_limit_date` | Recorded after purchase (leakage) |
| `order_delivered_customer_date`, `order_estimated_delivery_date` | Used only to build the targets and `promised_days` |
| `order_status` | Constant after the population filter |
| `payment_sequential`, `payment_value` | Used to pick the primary payment; value duplicates price + freight |
| All review data | Recorded after delivery (leakage) |

### 2.2 Row Reduction: Aggregate to One Row per Order

The prediction unit is the order, so item and payment rows are aggregated to one row per order. This also prevents leakage: items of one order cannot fall on both sides of the train/test split.

Items:
- Multi-seller orders take the **furthest** seller's distance, assuming the order is complete when the last parcel arrives.
- `same_state` holds only if every seller is in the customer's state.
- `seller_state` is the state of the **furthest** seller, the same seller that defines `distance_km`, so both location features describe the seller that determines delivery time. If no item has a distance, the first item's seller is used.
- `product_category` is the category of the **largest** item by volume, the item most likely to need special packing, handling or shipping. If volumes are tied or missing, the first item is used.
- Weight and volume stay missing only if no item has a measurement.

In [12]:
grouped = order_items.groupby("order_id")
item_features = grouped.agg(
    n_items=("order_item_id", "size"),
    n_products=("product_id", "nunique"),
    n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    distance_km=("distance_km", "max"),
    same_state=("same_state", "all"),
    customer_state=("customer_state", "first"),
)
item_features["total_weight_g"] = grouped["product_weight_g"].sum(min_count=1)
item_features["total_volume_cm3"] = grouped["volume_cm3"].sum(min_count=1)
furthest_seller_state = (
    order_items.sort_values(["distance_km", "order_item_id"], ascending=[False, True], na_position="last")
    .drop_duplicates("order_id").set_index("order_id")["seller_state"]
)
largest_item_category = (
    order_items.sort_values(["volume_cm3", "order_item_id"], ascending=[False, True], na_position="last")
    .drop_duplicates("order_id").set_index("order_id")["product_category"]
)
item_features = item_features.join(furthest_seller_state).join(largest_item_category)

print(f"Item rows {len(order_items):,} aggregated into {len(item_features):,} orders")

Item rows 110,189 aggregated into 96,470 orders


Payments: the primary payment type is the order's lowest-numbered payment row. This is usually `payment_sequential == 1`, but 78 orders have payment rows numbered from 2, so taking the lowest number keeps their real payment type instead of marking it `unknown`. The highest instalment count across an order's payments is kept.

In [13]:
payment_features = pd.DataFrame({
    "payment_type": (payments.sort_values("payment_sequential").drop_duplicates("order_id")
                     .set_index("order_id")["payment_type"]),
    "max_installments": payments.groupby("order_id")["payment_installments"].max(),
})
print(f"Payment rows {len(payments):,} aggregated into {len(payment_features):,} orders")

Payment rows 103,886 aggregated into 99,440 orders


## 3. Data Transformation

### 3.1 Feature Construction

New attributes built from existing ones. The features are intended to be available immediately after checkout and payment submission; their source timing is part of the remaining leakage audit. The two targets are known only after delivery.

| Constructed attribute | Built from | Section |
|---|---|---|
| `is_late` (target) | delivered calendar date > estimated calendar date; same-date delivery is on time | 3.1 |
| `delivery_days` (target) | delivered date - purchase time, in fractional days | 3.1 |
| `promised_days` | estimated delivery date - purchase time (the promise shown at checkout) | 3.1 |
| `purchase_hour`, `purchase_dayofweek`, `purchase_month` | purchase timestamp | 3.1 |
| `distance_km`, `same_state` | customer and seller ZIP centroids | 2.1 |
| `volume_cm3` | product length x height x width | 2.1 |
| `n_items`, `n_products`, `n_sellers`, `total_price`, `total_freight`, `total_weight_g`, `total_volume_cm3` | aggregation of items | 2.2 |
| `max_installments` | aggregation of payments | 2.2 |

In [14]:
def days_between(later, earlier):
    return (later - earlier).dt.total_seconds() / 86_400


purchased = orders["order_purchase_timestamp"]

# Targets: post-purchase timestamps are used here only.
orders["is_late"] = (orders["order_delivered_customer_date"].dt.normalize()
                     > orders["order_estimated_delivery_date"].dt.normalize()).astype(int)
orders["delivery_days"] = days_between(orders["order_delivered_customer_date"], purchased)

# Purchase-time features.
orders["promised_days"] = days_between(orders["order_estimated_delivery_date"], purchased)
orders["purchase_hour"] = purchased.dt.hour
orders["purchase_dayofweek"] = purchased.dt.day_name()
orders["purchase_month"] = purchased.dt.month_name()

Assemble the order table from the order-level, item and payment attributes:

In [ ]:
NUMERIC_FEATURES = [
    "promised_days", "distance_km", "same_state", "purchase_hour",
    "n_items", "n_products", "n_sellers", "total_price", "total_freight",
    "total_weight_g", "total_volume_cm3", "max_installments",
]
CATEGORICAL_FEATURES = [
    "customer_state", "seller_state", "product_category", "payment_type",
    "purchase_dayofweek", "purchase_month",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGETS = ["is_late", "delivery_days"]

order_table = (
    orders.set_index("order_id")
    .join(item_features)
    .join(payment_features)
    .reset_index()
)
order_table["payment_type"] = order_table["payment_type"].fillna("unknown")  # Section 1.1
order_table["same_state"] = order_table["same_state"].astype(int)  # Section 3.4
order_table = order_table[["order_id", "order_purchase_timestamp", "order_delivered_customer_date", *FEATURES, *TARGETS]]

order_table = order_table.rename(columns={"order_delivered_customer_date": "outcome_available_at"})
assert order_table["order_id"].is_unique
print(f"Order table: {len(order_table):,} orders x {len(FEATURES)} features + {len(TARGETS)} targets")
order_table.head()

### 3.2 Normalisation

**No scaling is applied in this notebook; these are decisions for the model Pipelines.**

Numeric features sit on very different scales (hours 0-23 vs. price up to thousands), and several are right-skewed.

| Model family | Normalisation | Why |
|---|---|---|
| A: Linear | `StandardScaler` on numeric features, inside the `Pipeline` | Coefficients and regularisation depend on scale |
| B: Tree-based, C: Gradient boosting | None | Splits are unaffected by scale |
| Regression target, linear models | `log1p(delivery_days)` via `TransformedTargetRegressor` | Reduces the right skew and the pull of long deliveries without removing them (Section 1.2) |

Scaling parameters are fitted on training folds only, so nothing is fitted here. The table below is the evidence for these choices.

In [16]:
scale_view = order_table[NUMERIC_FEATURES + ["delivery_days"]].agg(["min", "max", "std", "skew"]).T
scale_view.loc["log1p(delivery_days)"] = np.log1p(order_table["delivery_days"]).agg(["min", "max", "std", "skew"])
scale_view

,min,max,std,skew
promised_days,2.01,155.14,8.76,0.91
distance_km,0.00,3398.55,592.67,1.65
same_state,0.00,1.00,0.48,0.59
purchase_hour,0.00,23.00,5.33,-0.60
n_items,1.00,21.00,0.54,7.56
n_products,1.00,8.00,0.23,8.07
n_sellers,1.00,5.00,0.12,9.87
total_price,0.85,13440.00,209.05,9.89
total_freight,0.00,1794.96,21.56,12.28
total_weight_g,0.00,184400.00,4770.11,6.57


### 3.3 Discretisation

**Not applied to the shared data.** All continuous features keep their raw values here, because the effect of discretisation depends on the model family:

| Model family | Discretisation | Why |
|---|---|---|
| B: Tree-based, C: Gradient boosting | None | Trees choose their own cut points from the data, which fixed bins cannot beat; binning would only discard detail (e.g. 90 km and 300 km in the same bin) |
| A: Linear | Optional, inside the `Pipeline` | A linear model fits one straight line per feature, but distance has a curved effect (see the table below: moving from the first sextile (under 86 km) to the second (86-306 km) adds 3.4 days, while moving to the last sextile (965-3,399 km) adds only 4.2 days; each extra kilometre matters far more at short distances). Binning distance with `KBinsDiscretizer` and one-hot encoding the bins lets the model learn a separate effect per range. Bin edges are learned from training folds only. A `log1p` transform is an alternative that keeps the values continuous |

Whether the linear models use binning is decided in the model notebooks, by comparing cross-validation results with and without it.

Discretisation is used for **analysis only**: distance sextiles provide a comparison with the corrected Phase 1 check (Dashboard Figure 12). With same-date delivery counted on time, the late rate ranges from 4.4% to 10.2% across these sextiles while delivery time roughly triples. Phase 1 reported delivery time in whole days (`.dt.days`, which drops the fractional part), so its means are about 0.46 days lower than the fractional `delivery_days` used here.

The classification target itself is a discretisation of the delivery outcome: delivery time relative to the promise, cut at zero into late and on time.

In [17]:
by_distance = order_table.groupby(pd.qcut(order_table["distance_km"], 6), observed=True)
pd.DataFrame({
    "late rate (%)": by_distance["is_late"].mean() * 100,
    "delivery days (fractional)": by_distance["delivery_days"].mean(),
    "delivery days (whole, Phase 1)": by_distance["delivery_days"].apply(lambda s: np.floor(s).mean()),
})

,late rate (%),delivery days (fractional),"delivery days (whole, Phase 1)"
distance_km,,,
"(-0.001, 86.103]",4.40,6.39,5.93
"(86.103, 306.456]",5.21,9.79,9.33
"(306.456, 435.562]",6.84,12.38,11.91
"(435.562, 629.124]",6.80,13.05,12.58
"(629.124, 965.426]",7.14,14.72,14.25
"(965.426, 3398.548]",10.17,18.96,18.50


### 3.4 Encoding

- **Binary attributes** (`same_state`, `is_late`) are encoded as 0/1 here. This is a fixed mapping with nothing learned from the data.
- **Categorical attributes** stay as text here and are encoded inside each model `Pipeline`, so the category list comes from training folds only:

| Model family | Encoding | Why |
|---|---|---|
| A: Linear | One-hot (`handle_unknown="ignore"`) | No false ordering between categories |
| B: Tree-based | Ordinal | Trees split on codes; one-hot would spread a category across many sparse columns |
| C: CatBoost | Native categorical handling | Ordered target statistics avoid target-encoding leakage (Prokhorenkova et al., 2018) |

Cardinality stays manageable after dropping cities and ZIP prefixes (Section 2.1): about 145 one-hot columns in total.

- **Cyclical attribute** `purchase_hour` (0-23) stays as a number here. Hours 23 and 0 are adjacent in reality and behave alike (late-night orders take longest), but as numbers they are furthest apart:

| Model family | Encoding of `purchase_hour` | Why |
|---|---|---|
| A: Linear | Sine/cosine pair: `sin(2π·hour/24)`, `cos(2π·hour/24)`, inside the `Pipeline` | A single numeric hour forces a straight-line effect; the pair places 23:00 next to 0:00. A fixed formula, so nothing is learned from the data |
| B: Tree-based, C: Gradient boosting | Raw hour | Trees can split at both ends (e.g. hour <= 1 and hour >= 22) |

`purchase_month` is also cyclical but is one-hot encoded as a category, so it has no false ordering.

In [18]:
binary = ["same_state", "is_late"]
assert order_table[binary].isin([0, 1]).all().all()

order_table[CATEGORICAL_FEATURES].nunique().rename("distinct values").to_frame()

,distinct values
customer_state,27
seller_state,22
product_category,72
payment_type,5
purchase_dayofweek,7
purchase_month,12


## 4. Feature Engineering Strategy

The design principles behind the feature set are listed at the top of this notebook.

**Final feature set**

| Group | Features | Phase 1 / literature evidence |
|---|---|---|
| Promise | `promised_days` | Promise window is non-stationary and tracks lagging performance (Fig. 23); customers judge delivery against the promise (Deshpande and Pendem, 2023) |
| Route | `distance_km`, `same_state`, `customer_state`, `seller_state` | Delivery time triples across distance sextiles; corrected calendar-date late rate ranges from 4.4% to 10.2% (Fig. 12a/b); Haversine distance (de Araujo and Etemad, 2021) |
| Timing | `purchase_hour`, `purchase_dayofweek`, `purchase_month` | Friday orders take 14% longer than Sunday (Fig. 27), effect in the handling stage (Fig. 21); Black Friday peak (Fig. 1); order timing reflects cut-offs (Salari et al., 2022) |
| Order complexity | `n_items`, `n_products`, `n_sellers` | Multi-seller and multi-item orders differ in delivery time (Fig. 28); complexity features (Salari et al., 2022) |
| Order value and size | `total_price`, `total_freight`, `total_weight_g`, `total_volume_cm3`, `product_category` | Weight only weakly related (Fig. 29), kept for pruning by importance; freight-to-price ratio showed a flat late rate (Fig. 22), so the ratio is not built |
| Payment | `payment_type`, `max_installments` | Boleto adds about 1.2 days in the processing stage (Figs. 15a, 16) |

**Considered and deferred:** seller history (past late rate) and marketplace backlog. Phase 1 found backlog explains delay better than concurrent volume (Figs. 17-19). Both need time-aware computation (only orders before each purchase, and statistics from training data only) to avoid leakage, so they are left as possible extensions.

**Excluded on purpose:** `order_approved_at` and `shipping_limit_date` (set after purchase), `order_delivered_carrier_date` and `order_delivered_customer_date` (the outcome), `order_status`, and all review data (after delivery).

**Leakage check:**

In [ ]:
POST_PURCHASE_COLUMNS = {
    "order_status", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "shipping_limit_date", *TARGETS,
}
POST_PURCHASE_KEYWORDS = ("approved", "carrier", "delivered", "review", "late", "delivery_days")

leaked = [c for c in FEATURES
          if c in POST_PURCHASE_COLUMNS or any(k in c for k in POST_PURCHASE_KEYWORDS)]
assert not leaked, f"Post-purchase columns used as features: {leaked}"
print("Post-purchase column-name screen passed; full source-timing and model-pipeline audit remains open.")

Post-purchase column-name screen passed; full source-timing and model-pipeline audit remains open.


**Correlation check (multicollinearity)**

The attributes do not need to be independent: the tree-based and gradient boosting families are unaffected by correlated inputs. Correlation matters in two places:
- **Linear models:** highly correlated features make individual coefficients unstable and hard to interpret (predictions are still fine). Regularisation (Ridge, L2 logistic regression) reduces this.
- **Feature importance:** correlated features share importance, so each can look weak on its own. They should be interpreted as a group.

Only **perfectly redundant** features are removed. `purchase_is_weekend` was dropped because it is fully determined by `purchase_dayofweek` (Saturday or Sunday). After one-hot encoding it equals the sum of two day columns, which is perfect collinearity for linear models and adds no information for trees.

The pairs below (|Spearman| >= 0.4) are correlated but carry different meanings, so they are kept:

| Pair | Why both are kept |
|---|---|
| `total_weight_g`, `total_volume_cm3` | Freight is priced on both actual weight and volume |
| `distance_km`, `same_state` | Crossing a state border can add a hand-off beyond the extra distance |
| `promised_days`, `distance_km` / `same_state` | The promise only partly reflects distance; the gap between them is informative |
| `n_items`, `n_products`, `n_sellers` | More items, more distinct products and more sellers are different kinds of complexity |
| `total_price`, `total_freight`, `total_weight_g`, `total_volume_cm3` | Value and physical size overlap but are not interchangeable |
| `total_freight`, `distance_km` / `same_state` / `promised_days` | Freight is priced partly on distance, but also reflects weight and carrier pricing |

In [20]:
spearman = order_table[NUMERIC_FEATURES].corr(method="spearman")
pairs = [
    (a, b, spearman.loc[a, b])
    for i, a in enumerate(NUMERIC_FEATURES) for b in NUMERIC_FEATURES[i + 1:]
    if abs(spearman.loc[a, b]) >= 0.4
]
pd.DataFrame(pairs, columns=["feature A", "feature B", "Spearman"]).sort_values(
    "Spearman", key=abs, ascending=False, ignore_index=True)

,feature A,feature B,Spearman
0,total_weight_g,total_volume_cm3,0.78
1,distance_km,same_state,-0.76
2,n_products,n_sellers,0.63
3,promised_days,distance_km,0.62
4,distance_km,total_freight,0.59
5,promised_days,same_state,-0.59
6,same_state,total_freight,-0.56
7,n_items,n_products,0.55
8,total_price,total_weight_g,0.54
9,total_freight,total_weight_g,0.49


**Missing values left for the Pipeline imputer (Section 1.1):**

In [21]:
missing = order_table[FEATURES].isna().sum()
missing = missing[missing > 0]
pd.DataFrame({"missing": missing, "share (%)": missing / len(order_table) * 100})

,missing,share (%)
distance_km,477,0.49
total_weight_g,16,0.02
total_volume_cm3,16,0.02
max_installments,1,0.00


**Targets:**

In [22]:
late_rate = order_table["is_late"].mean()
print(f"is_late: {late_rate:.2%} late, imbalanced. Always predicting 'on time' already scores {1 - late_rate:.2%} accuracy.")
order_table["delivery_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_frame().T

is_late: 6.77% late, imbalanced. Always predicting 'on time' already scores 93.23% accuracy.


,count,mean,std,min,25%,50%,75%,95%,99%,max
delivery_days,96470.00,12.56,9.55,0.53,6.77,10.22,15.72,29.27,46.05,209.63


## 5. Train/Test Splits and Validation Folds

### 5.1 Primary: chronological holdout; secondary: random benchmark

The prediction is made when an order is purchased. A fixed **2018-05-26** purchase-date cutoff holds out approximately the last 20% of this delivered-order population. Earlier purchases train the primary models only when their delivery outcomes were already known before the cutoff; earlier purchases delivered later are `unavailable_at_cutoff` and excluded from the primary fit. The same partitions serve classification and regression. The later test set is used once for final evaluation.

The teammate's original seed-42 stratified random 80/20 split is retained as `random_split` for a separate secondary benchmark. It asks how well models fit the same historical mix; it is not evidence of future-period performance. Never mix its training rows with the primary experiment. The two experiments need separate model fits. Report the random result as context, not as evidence of future-period performance.


In [ ]:
# A whole-day boundary avoids splitting orders from one calendar date across periods.
assert order_table["order_purchase_timestamp"].notna().all()
assert order_table["outcome_available_at"].notna().all()
purchased_before = order_table["order_purchase_timestamp"] < TIME_CUTOFF
label_known = order_table["outcome_available_at"] < TIME_CUTOFF
order_table["split"] = np.select(
    [~purchased_before, purchased_before & label_known],
    ["test", "train"],
    default="unavailable_at_cutoff",
)

_, random_test_ids = train_test_split(
    order_table["order_id"], test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=order_table["is_late"],
)
order_table["random_split"] = np.where(order_table["order_id"].isin(random_test_ids), "test", "train")

assert order_table.loc[order_table["split"].eq("train"), "outcome_available_at"].max() < TIME_CUTOFF
assert order_table.loc[order_table["split"].eq("test"), "order_purchase_timestamp"].min() >= TIME_CUTOFF
print("Primary cutoff:", TIME_CUTOFF.date())
print(order_table.groupby("split").agg(
    orders=("order_id", "size"),
    late_rate_pct=("is_late", lambda s: s.mean() * 100),
    mean_delivery_days=("delivery_days", "mean"),
))
print("\nSecondary random benchmark:")
print(order_table.groupby("random_split").agg(
    orders=("order_id", "size"),
    late_rate_pct=("is_late", lambda s: s.mean() * 100),
    mean_delivery_days=("delivery_days", "mean"),
))


Primary cutoff: 2018-05-26
                       orders  late_rate_pct  mean_delivery_days
split                                                           
test                    19363           3.48                8.77
train                   75099           7.10               13.29
unavailable_at_cutoff    2008          26.20               21.85

Secondary random benchmark:
              orders  late_rate_pct  mean_delivery_days
random_split                                           
test           19294           6.77               12.53
train          77176           6.77               12.56


### 5.2 Validation folds

**Primary:** reserve the earliest 40% of primary training orders as initial history (`cv_fold = -1`). The next 60% form five successive validation blocks (`cv_fold = 0..4`). For each block, fit on earlier purchases **whose outcomes were known before that block starts**. Some older purchases are therefore unavailable to an earlier fold even though they are known by the final holdout date. Use the explicit `(fit, validation)` indices in the loading example; `PredefinedSplit` would incorrectly include later blocks in training.

**Secondary:** `random_cv_fold = 0..4` stores five seeded stratified folds on `random_split == train`. These can be used with `PredefinedSplit` for the separate random benchmark. Both tasks share assignments, but their metrics are evaluated separately. Fit preprocessing and threshold selection only within each experiment's training folds.


In [ ]:
# Validation blocks are chronological, with an initial history for the first fit.
order_table["cv_fold"] = pd.Series(pd.NA, index=order_table.index, dtype="Int64")
primary_train = order_table.loc[order_table["split"].eq("train")].sort_values(
    ["order_purchase_timestamp", "order_id"]
)
initial_n = int(len(primary_train) * 0.4)
order_table.loc[primary_train.index[:initial_n], "cv_fold"] = -1
for fold, block in enumerate(np.array_split(primary_train.index[initial_n:].to_numpy(), N_FOLDS)):
    order_table.loc[block, "cv_fold"] = fold
    start = order_table.loc[block, "order_purchase_timestamp"].min()
    fit = primary_train.loc[
        (primary_train["order_purchase_timestamp"] < start) &
        (primary_train["outcome_available_at"] < start)
    ]
    assert len(fit) > 0 and fit["order_purchase_timestamp"].max() < start
    assert fit["outcome_available_at"].max() < start

order_table["random_cv_fold"] = pd.Series(pd.NA, index=order_table.index, dtype="Int64")
random_train = order_table.loc[order_table["random_split"].eq("train")]
folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for fold, (_, val_idx) in enumerate(folds.split(random_train, random_train["is_late"])):
    order_table.loc[random_train.index[val_idx], "random_cv_fold"] = fold

assert order_table.loc[order_table["split"].eq("train"), "cv_fold"].notna().all()
assert order_table.loc[~order_table["split"].eq("train"), "cv_fold"].isna().all()
assert order_table.loc[order_table["random_split"].eq("train"), "random_cv_fold"].notna().all()
assert order_table.loc[order_table["random_split"].eq("test"), "random_cv_fold"].isna().all()
print(order_table.loc[order_table["split"].eq("train")].groupby("cv_fold").agg(
    orders=("order_id", "size"), late_rate_pct=("is_late", lambda s: s.mean() * 100),
))


         orders  late_rate_pct
cv_fold                       
-1        30039           3.65
0          9012          11.71
1          9012           5.50
2          9012          11.93
3          9012          16.52
4          9012           1.38


### 5.3 Save Outputs

- `data/phase2_order_table.csv`: one row per order with features, targets, outcome availability metadata, both splits and both fold assignments.
- `data/phase2_feature_spec.json`: feature and target lists plus the fixed primary cutoff and split definitions.


In [ ]:
order_table.to_csv(OUTPUT_TABLE, index=False)
spec = {
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "classification_target": "is_late",
    "regression_target": "delivery_days",
    "random_state": RANDOM_STATE,
    "n_folds": N_FOLDS,
    "primary_cutoff": str(TIME_CUTOFF.date()),
    "primary_split": "split",
    "primary_fold": "cv_fold",
    "secondary_split": "random_split",
    "secondary_fold": "random_cv_fold",
}
OUTPUT_SPEC.write_text(json.dumps(spec, indent=2))
print(f"Saved {OUTPUT_TABLE} ({len(order_table):,} rows) and {OUTPUT_SPEC}")

Saved /Users/joshualum/Documents/VSCode/IT5006_Grp_6/data/phase2_order_table.csv (96,470 rows) and /Users/joshualum/Documents/VSCode/IT5006_Grp_6/data/phase2_feature_spec.json


## Summary

| Stage | Outcome |
|---|---|
| Cleaning | 96,470 delivered orders; categorical gaps filled with `unknown`, numeric gaps left for Pipeline imputation; invalid geolocation points removed, outliers kept |
| Reduction | Location attributes replaced by distance and same-state; item rows aggregated to one row per order |
| Transformation | Targets and purchase-time features constructed; fitted scaling, target transformation and encoding remain inside model Pipelines |
| Feature engineering | 18 purchase-time features (12 numeric, 6 categorical); post-purchase fields excluded from model inputs |
| Primary evaluation | Fixed calendar-date cutoff near the 80th purchase percentile; later orders held out; earlier orders train only when their outcomes were known by the cutoff; five expanding validation windows |
| Secondary evaluation | Original seeded stratified random 80/20 split and five stratified folds, run as a separate same-history benchmark |

Report both test periods' class rates and target distributions, and inspect primary test performance by purchase month. The primary later-period score supports a within-Olist future-order claim; the random benchmark answers a different question and must not drive primary model selection. Neither estimates performance beyond the historical 2016–2018 Olist data. The delivered-only population excludes orders with no recorded delivery, including cancellations and stock unavailability, so the scores cannot be interpreted as unconditional risk for every new purchase. The dataset end can bias later purchases toward faster completions. The fixed 2018-05-26 cutoff does not depend on when the analysis is run. An end-to-end leakage audit is still needed before model performance is reported.
